# Machine learning training

------------------------------------------

### Initializing the work environment

```bash
python -m venv venv
source venv/bin/activate
pip install -r requirement.txt
```

### Import the necessary libs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.decomposition import NMF
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

-------------------------------------------------------------

The dataset consists of 3 columns: ID, a secret color, and the testimony. The color will be used to check if our clusters are accurate at the end of the learning phase of our project.

Our objective is to group similar testimonies into clusters, where each cluster potentially corresponds to a specific diagnosis or underlying disease.

In [ ]:
# Function to load the dataset
def init(path):
    # Load the dataset with pandas, mapping columns to 0 (ID), 1 (Color), and 2 (Testimony)
    df = pd.read_csv(path, header=None, names=[0, 1, 2])
    # Print the shape to ensure the dataset is loaded perfectly
    print(f"Dataset loaded perfectly : {df.shape}")
    return df

df = init("./cleaned_dataset.csv")

In [ ]:
# Function to print the dataset (useful for debug or visualization purpose)
def print_df(n, df):
    # Grab the first 'n' rows of the dataset
    df_print = df.head(n)
    print(f"These are the first {n} lines of the dataset:\n\n{df_print}")

----------------------------------------

## Training with cleaned dataset

### First step : Vectorize the dataset 

In [ ]:
# This function will vectorize the text, meaning it will put the words we are truly interested in into a matrix 
# and assign them a value based on their presumed importance in the dataset (TF-IDF protocol).
def vectorize_text(df):
    # This line configures the rules to define the importance of the words so that the training can proceed correctly
    vectorizer = TfidfVectorizer(stop_words='english')
    
    # This line creates the matrix and transforms the text into mathematical values
    X_tfidf = vectorizer.fit_transform(df[2])
    
    # We use TruncatedSVD (LSA) to reduce dimensionality to 50 parameters, making it easier for the model to learn
    svd = TruncatedSVD(n_components=50, random_state=42)
    X_svd = svd.fit_transform(X_tfidf)
    
    # This line retrieves the actual words kept by the vectorizer (useful to debug and check our vocabulary)
    kept_words = vectorizer.get_feature_names_out()
    
    # Visualizing the number of words kept by our code
    print(f"Number of highly relevant medical words kept: {len(kept_words)}")

    return X_tfidf, X_svd, vectorizer

X_tfidf, X_svd, vectorizer = vectorize_text(df)

Okay, now we are going to show the useful words kept by the function and try to understand how to find the different clusters.

Extremely simplified example of matrix X :

|  | Col n0 : "ache" | Col n1 : "chest" |
|:-------- |:--------:| --------:|
| Patient 1| 0.0   | 0.2    |
| Patient 2| 0.4   | 0.1    |

In [ ]:
# X: TF-IDF matrix, vectorizer: The fitted TfidfVectorizer object (our vocabulary/dictionary)
def plot_top_tfidf_words(X, vectorizer, top_n=20):

    # This line calculates the sum of all values in each column (the total TF-IDF score for each word across all documents)
    tfidf_sum = np.array(X.sum(axis=0)).flatten()
    
    # This line retrieves the actual readable words (feature names) corresponding to the matrix columns
    words = vectorizer.get_feature_names_out()
    
    # This sorts the array and extracts the indices of the top 20 words with the highest overall scores
    top_indices = tfidf_sum.argsort()[-top_n:]
    
    # This extracts the actual words and their corresponding scores using the sorted indices
    top_words = [words[i] for i in top_indices]
    top_scores = tfidf_sum[top_indices]
    
    # Graphic setup
    plt.figure(figsize=(10, 8))
    plt.barh(top_words, top_scores, color='coral')
    plt.title(f"Top {top_n} most 'discriminant' words")
    plt.xlabel("Score")
    plt.ylabel("Medical words")
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

plot_top_tfidf_words(X_tfidf, vectorizer)

Memo english word to french :
- sore = Endolori (qui fait mal)
- ache = Douleur
- cough = Toux
- sharp = Aigu / Vif

Interpretation of the "most discriminative words":

We have some body parts, which will be very useful to give a diagnosis. The body parts we have are: chest, skin, ear, and throat.
We also have some feelings (symptoms) about them: pain (ache), cold, sharp, long, sore, and cough (cough is not a feeling, but a good clue to give a diagnosis).
So, we can now try to understand and think about a way to train an AI to give a diagnosis based on a testimony. For example: if the testimony has "cough" and "throat" in it, it could be tonsillitis (maux de gorge).

---------------------------------

## Cloud Analysis and cluster reflection

Now, we will see how to group the different lines of our dataset into clusters and determine the optimal number of clusters we need.

### First step : Try with the K-means machine learning algorithm

-------------------------------------

The Elbow Method helps us find the perfect number of clusters. We test the algorithm with 1, 2, 3... up to 12 clusters and calculate the "inertia" (which measures how tight our groups are). Then, we plot these values on a graph.

The curve looks like an arm. The best number of clusters is exactly at the "elbow", where the line stops dropping fast and becomes almost flat. Adding more clusters after this point doesn't improve the result.

In [ ]:
def find_optimal_k_combined(X, max_k=20):
    inertias = []
    scores = [] # Only for K >= 2 (Silhouette score requires at least 2 clusters to compute distances)

    K_range = range(1, max_k + 1)
    
    print("Calculating scores...")
    
    for k in K_range:
        # Initialize K-Means model for the current k
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        kmeans.fit(X)
        
        # Store the inertia (used for the Elbow method)
        inertias.append(kmeans.inertia_)
        
        # We only calculate the silhouette score if K > 1
        if k > 1:
            score = silhouette_score(X, kmeans.labels_)
            scores.append(score)
            
    # Graph creation
    fig, ax1 = plt.subplots(figsize=(15, 8))

    # Left axis: INERTIA (Elbow Method)
    color1 = 'teal'
    ax1.set_xlabel('Number of clusters (K)', fontsize=14)
    ax1.set_ylabel('Inertia (Intra-cluster distance)', color=color1, fontsize=14)
    ax1.plot(K_range, inertias, marker='o', linestyle='-', color=color1, markersize=8, label="Inertia (Elbow)")
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.set_xticks(K_range)
    ax1.grid(True, linestyle='--', alpha=0.5)

    # Right axis: SILHOUETTE SCORE
    ax2 = ax1.twinx() # This creates a second Y-axis sharing the same X-axis
    color2 = 'purple'
    ax2.set_ylabel('Silhouette Score (Higher is better)', color=color2, fontsize=14)
    
    # Note: 'scores' list has one less value than 'inertias' (no K=1), so we map it to range(2, max_k + 1)
    K_range_sil = range(2, max_k + 1)
    ax2.plot(K_range_sil, scores, marker='s', linestyle='--', color=color2, markersize=8, label="Silhouette Score")
    ax2.tick_params(axis='y', labelcolor=color2)

    plt.title("Combined Metrics: Elbow Method & Silhouette Score", fontsize=16, weight='bold')
    fig.tight_layout()
    plt.show()

find_optimal_k_combined(X_svd, max_k=20)

To interpret the Elbow Method graph, we must identify a significant change in the slope of the curve. Here, we can observe a very clear 'elbow' at K=10 and K=8 and K=18, which gives us the optimal number of clusters for our dataset.
I decided to add another metric to base our judgment on: the Silhouette score. In theory, the higher the Silhouette score, the better the clusters. Therefore, if we hesitate between two potential elbows, we can cross-reference with this score to make the best choice. In this case, we can clearly see that K=10 forms a distinct elbow and simultaneously has the best Silhouette score.

As we can observe, we do not simply select the lowest possible inertia. We must find a balance between a low inertia and a reasonable number of clusters. For instance, creating 1,000 clusters for 1,000 patients would result in an inertia of 0, but this would lead to overfitting and defeat the whole purpose of clustering.

In [ ]:
def apply_kmeans(X, n_clusters):
    # 1. K-Means Training phase
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(X)
    # Retrieve the cluster assignments for each patient
    labels_predict = kmeans.labels_
    
    # 2. Dimensionality reduction with t-SNE (compressing our 50D SVD space into 2D for plotting)
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X) 
    
    # Graphic setup
    plt.figure(figsize=(15, 8))
    # Create a background contour based on clusters
    plt.tricontourf(X_tsne[:, 0], X_tsne[:, 1], labels_predict, cmap='tab10', alpha=0.2)
    # Plot the actual patient points
    plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_predict, cmap='tab10', alpha=0.6, s=50)
    
    # Loop through each cluster to draw its center (centroid)
    for i in range(n_clusters):
        # Filter points belonging to the current cluster
        cluster_points = X_tsne[labels_predict == i]
        
        # Calculate the center (mean of X coordinates and mean of Y coordinates)
        center_x = cluster_points[:, 0].mean()
        center_y = cluster_points[:, 1].mean()
        
        # Draw a large cross with a black border to represent the center
        plt.scatter(center_x, center_y, c='black', marker='X', s=200, edgecolors='black', linewidths=1, zorder=2)

    plt.title("Cloud Point (K-Means Clustering)")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()
    
    return kmeans

In [ ]:
kmeans = apply_kmeans(X_svd, n_clusters=10)

def display_top_words_per_cluster(kmeans, X_tfidf, X_svd, vectorizer, n_mots=5):
    # We retrieve our vocabulary/dictionary to translate matrix indices back into real words
    words = vectorizer.get_feature_names_out()
    labels = kmeans.labels_
    
    # Calculate and print the overall Silhouette score for this model
    score = silhouette_score(X_svd, labels)
    print(f"Silhouette score : {score:.4f}\n")

    # Convert the sparse TF-IDF matrix into a dense array to calculate means easily
    X_dense = X_tfidf.toarray()
    
    # Loop through every cluster identified by K-Means
    for i in range(kmeans.n_clusters):
        # Isolate the patients belonging to cluster 'i'
        patients = X_dense[labels == i]
        
        if len(patients) > 0:
            # Calculate the average TF-IDF score for each word across all patients in this cluster
            mean_word = patients.mean(axis=0)
            # Sort the scores and grab the indices of the top 'n_mots' words (reversed to get highest first)
            top_indices = mean_word.argsort()[-n_mots:][::-1]
            # Map the indices back to actual readable words
            top_words = [words[ind] for ind in top_indices]
            
            # Print the cluster ID, its size, and its defining words
            print(f"Cluster {i} ({len(patients)} patients) : {', '.join(top_words)}")

# We call the function to display the top 6 defining words for each disease cluster
display_top_words_per_cluster(kmeans, X_tfidf, X_svd, vectorizer, n_mots=6)

Here, after the learning algorithm identifies the optimal clusters, we reduce the dimensionality of our data to 2 parameters (2D) to plot it on a graph. However, as we can see, it is hard to tell which points belong together. To fix this, we will color the points into different clusters based on their similarities and add a cross which is represent the center of the clusters we have.

As we can observe on the scatter plot, the result with 10 clusters is quite good. Nearly all points of the same color stay closely grouped together. While natural noise remains, it is clearly a highly satisfactory result. Furthermore, if we analyze the most discriminant words for each group, we can see that they perfectly describe distinct medical conditions. This proves that our clusters are not only mathematically robust, but they also make perfect clinical sense.

Here is the medical interpretation of the 10 identified clusters:

- Cluster 0 - Upper Body Orthopedics (shoulder, arm, elbow, lifting): Highlights joint and muscle pain localized in the upper limbs.

- Cluster 1 - Lower Body Orthopedics (knee, leg, hip, deep ache): Groups lower limb joint pain and mobility issues.

- Cluster 2 - Diabetes / Metabolic Issues (blood, thirsty, weight, losing, sugar): A textbook description of diabetes symptoms, including severe thirst and sudden weight loss.

- Cluster 3 - Orthostatic Hypotension / Anemia (pale, dizzy, stand, headaches): Captures the specific sensation of dizziness and paleness when standing up.

- Cluster 4 - Ear Infection / Otitis (ear, pain, fever, hear): Groups symptoms related to earaches, fever, and mild hearing difficulties.

- Cluster 5 - Localized Skin Inflammation (hot, red, sore, area): Describes acute, localized skin issues such as abscesses, burns, or local infections.

- Cluster 6 - Mild Cold / Viral Infection (nose, throat, mild, cough): Represents common, mild upper respiratory tract infections.

- Cluster 7 - Migraine (head, throbbing, vision): Perfectly isolates severe, throbbing headaches associated with visual disturbances.

- Cluster 8 - Asthma / Respiratory Distress (chest, short breath, tightness): Highlights severe lower respiratory issues, chest tightness, and breathing difficulties.

- Cluster 9 - Allergic Reaction (itchy, rash, hives, spots): Groups dermatological allergies, rashes, and urticaria (hives).

### Second step : Try with the DBScan machine learning algorithm

-------------------------------------

Now, we will try applying the DBSCAN algorithm to our dataset. DBSCAN is a highly unsuitable model for this specific task because it looks for dense clusters that are clearly separated from one another by empty spaces. As we have seen with our K-Means results, our textual data is continuous and does not look like that at all. However, it is very instructive to observe how an algorithm behaves when its mathematical logic is not adapted to the nature of the data.

In [ ]:
def apply_dbscan(X_svd):
    print("1. Launching DBSCAN...")
    # eps and min_samples are values to help DBSCAN find clusters
    # eps is the search radius around a single patient 
    # min_samples is the minimum number of patients needed in that radius to form a cluster
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels_dbscan = dbscan.fit_predict(X_svd)
    
    # Here we figure out how many clusters DBSCAN found. 
    # If label -1 exists (noise), it doesn't count as a valid medical cluster, so we subtract 1.
    n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    
    # Count how many patients were rejected and labeled as noise (-1)
    n_noise = list(labels_dbscan).count(-1)
    total_patients = len(labels_dbscan)
    
    print(f"Number of clusters found: {n_clusters}")
    print(f"Noise: {n_noise} / {total_patients} ({(n_noise/total_patients)*100:.2f}%)")
    
    # Here we count the number of patients in each found cluster (excluding noise)
    for i in range(n_clusters):
        print(f"Cluster {i} size: {list(labels_dbscan).count(i)} patients")

    # Reduce dimensionality to 2D for plotting
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X_svd)
    
    # --- Graphic setup ---
    plt.figure(figsize=(12, 8))
    # Plot the points. Noise (-1) will automatically get a distinct color in tab20.
    scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_dbscan, cmap='tab20', alpha=0.7, s=50)
    plt.title("DBSCAN Failure: High density of Noise (-1) or Giant Cluster")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.show()

apply_dbscan(X_svd)

Unlike K-Means, DBSCAN attempts to find clusters based on data density rather than distance to a central point. Consequently, we cannot force it to generate a specific number of clusters. This makes it completely unsuitable for our situation. As we can see above, it only found 2 clusters. Therefore, there is no reason to explore this algorithm any further for this specific subject.

### Third step : Try with the Agglomerative machine learning algorithm

---------------------------------------------------

In [ ]:
def find_optimal_k_agglo_svd(X_svd, max_k=20):
    # This list will store the silhouette score calculated for each k
    silhouette_scores = []
    
    # We must start at 2, because silhouette score requires at least 2 clusters to compute distances
    K_range = range(2, max_k + 1)
    
    for k in K_range:
        # Initialize the Agglomerative model using the 'ward' linkage (minimizes variance inside clusters)
        agg_model = AgglomerativeClustering(n_clusters=k, linkage='ward')
        
        # X_svd is ALREADY a dense numpy array, so we train the model and get predictions directly
        labels = agg_model.fit_predict(X_svd)
        
        # We calculate the silhouette score on the reduced SVD space to measure cluster quality
        score = silhouette_score(X_svd, labels)
        silhouette_scores.append(score)
    
    # Graphic generation
    plt.figure(figsize=(15, 8))
    plt.plot(K_range, silhouette_scores, marker='o', linestyle='-', color='purple', markersize=8)
    
    plt.title("Silhouette Method to find the optimal K (Agglomerative on SVD)")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Silhouette Score (HIGHER is better)")
    plt.xticks(K_range)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

# Run the function using the previously generated X_svd matrix
find_optimal_k_agglo_svd(X_svd, max_k=20)

This graph displays the Silhouette score for the Agglomerative Clustering algorithm, tested across a range of 2 to 20 clusters. This metric is crucial for identifying the most efficient cluster count. Although the Silhouette score is fundamentally a theoretical mathematical indicator, it serves as our most robust evaluation method, especially considering the strict library constraints we had to work with for this model.

In [ ]:
def apply_agg(X_svd, n_clusters=12):
    # 1. Train the Agglomerative Clustering model
    agg_model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    labels_agg = agg_model.fit_predict(X_svd)
    
    # 2. Reduce to 2D with t-SNE for plotting
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X_svd)
    
    plt.figure(figsize=(15, 8))
    # We use the 'tab20' colormap because we have more than 10 clusters (12 clusters)
    plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_agg, cmap='tab20', alpha=0.6, s=50) 
    
    # Calculate and draw a visual "center" for each cluster (Even though Agglomerative doesn't use centroids natively)
    for i in range(n_clusters):
        cluster_points = X_tsne[labels_agg == i]
        
        if len(cluster_points) > 0:
            center_x = cluster_points[:, 0].mean()
            center_y = cluster_points[:, 1].mean()
            # Draw a red cross for the center
            plt.scatter(center_x, center_y, c='red', marker='X', s=400, edgecolors='black', linewidths=2, zorder=5)

    plt.title("Archipelago of Diseases (Agglomerative LSA : SVD + t-SNE)")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

    return labels_agg

In [ ]:
labels_agg = apply_agg(X_svd)

def display_top_words_per_cluster_agglo(labels_agg, X_tfidf, n_clusters=12):
    # Get the vocabulary to translate indices to actual words
    kept_words = vectorizer.get_feature_names_out()
    # Convert sparse TF-IDF matrix to dense array for calculations
    X_dense = X_tfidf.toarray()
    
    # Calculate and print the overall Silhouette score
    score = silhouette_score(X_svd, labels_agg)
    print(f"Silhouette score : {score:.4f}\n")
    
    # Loop through each cluster to find its defining symptoms
    for i in range(n_clusters):
        cluster_patients = X_dense[labels_agg == i]
        
        if len(cluster_patients) > 0:
            # Calculate the mean TF-IDF score of each word for this specific cluster
            mean_words = cluster_patients.mean(axis=0)
            # Grab the indices of the top 6 highest scoring words
            top_words_idx = mean_words.argsort()[-6:][::-1]
            # Map indices back to strings
            top_words = [kept_words[idx] for idx in top_words_idx]
            
            # Print the results
            print(f"Cluster {i} ({len(cluster_patients)} patients) : {', '.join(top_words)}")

display_top_words_per_cluster_agglo(labels_agg, X_tfidf)

As we can observe with this model, the data points are no longer dispersed; they form very distinct and cohesive clusters. When we look at the most discriminant words for each group, we can clearly see that the categorization of each patient's disease is even more accurate and refined than with the K-Means model. Here is the medical interpretation of these 12 specific clusters:

- Cluster 0 - Cardiopulmonary Emergency (chest, cough, breath, pressure, short, tight): Highlights severe respiratory distress, asthma attacks, or potential cardiac issues. Interestingly, this is the largest group, which mirrors real-life emergency room volumes.

- Cluster 1 - Diabetes / Metabolic (vision, doctor, blood, losing, sugar, weight): A perfect textbook description of diabetes symptoms (blood sugar issues, vision changes, sudden weight loss).

- Cluster 2 - Severe Headache / Migraine (pain, head, starts, throbbing, ache, dull): Focuses on cranial pain, specifically throbbing or dull headaches.

- Cluster 3 - Lower Body Orthopedics (knee, deep, ache, leg, thigh, hip): Groups major joint and muscle pain in the lower limbs (hips, thighs, knees).

- Cluster 4 - Mild Viral Infection (nose, throat, mild, cough, day, cold): Represents benign, mild upper respiratory tract infections (the common cold).

- Cluster 5 - Localized Skin Infection / Burn (hot, blister, touched, small, red, skin): Clearly describes acute skin lesions, such as blisters, burns, or local infections.

- Cluster 6 - Ear Infection / Otitis (ear, pain, slight, fever, cold, hear): Specifically isolates ENT issues related to earaches and minor hearing loss.

- Cluster 7 - Allergic Reaction (itchy, rash, new, spots, raised, hives): Perfectly groups dermatological allergies, including rashes and hives.

- Cluster 8 - Distal Upper Limb Orthopedics (wrist, elbow, palm, thumb, forearm): Highly specific cluster focusing on distal arm issues (tendinitis, carpal tunnel).

- Cluster 9 - Vagal Syncope / Hypotension (pale, dizzy, skin, stand, breath, heart): Captures the exact sensation of dizziness, paleness, and heart palpitations when standing up.

- Cluster 10 - Foot & Ankle Orthopedics / Podiatry (foot, toe, pushing, shoes, big, ankle): The most precise orthopedic cluster, focusing strictly on extremities and footwear-related pain.

- Cluster 11 - Proximal Upper Limb Orthopedics (shoulder, reaching, arm, overhead, lifting, sharp): Focuses on shoulder joint issues and pain triggered by lifting or overhead movements.

--------------------------------

## Conclusion

In conclusion, we have identified two viable models for clustering the multiple diseases that hospital patients present. However, the Agglomerative Clustering model proved to be the most effective and powerful. This is because the agglomerative approach can cluster closely related data more easily and with greater precision than K-Means. At the end of this unsupervised learning phase, we successfully isolated 12 distinct clusters, each representing a specific medical condition. Moving forward, incoming patients can be automatically evaluated against these clusters and directly routed to the appropriate hospital departments.